<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TTS_Voice_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TTS Voice Library — Curated Reference Clips for Voice Cloning

A browseable, searchable library of pre-licensed reference voice clips that work as inputs to the **voice-cloning tabs** in every TTS notebook in this suite (Higgs, MisoTTS, MOSS-TTS, Qwen3-TTS Base, dots.tts-soar, Fish S2 Pro, IndicF5).

Every entry has a **transcript** (required by all of the above) and is sourced from a public, permissive dataset or from the model's own demo. Browse in the Gradio UI, preview, then **download to local** to use as the reference in the matching notebook.

### What you get
- **12+ curated voices** across English, Chinese, French, German, Spanish, Hindi, Tamil, Kannada, Marathi, Punjabi, and more
- **Public-domain or CC-BY/MIT-licensed** material only — no voice-cloning consent gate, but every clip is from a published dataset where the speakers consented
- **Direct .wav download** to `/content/voices/<name>.wav` (one-click from the UI)
- **Optional Google Drive copy** so other notebooks in the suite can pick them up
- **Streaming HF datasets** (`LJSpeech`, `VCTK`) for additional English voices if you want more variety

### How to use with another TTS notebook
1. Run this notebook, browse the gallery, click **Download** on the voice you want
2. Switch to the TTS notebook (e.g. `Higgs-Audio_Colab.ipynb`), open the voice-cloning accordion
3. Upload the downloaded .wav, paste the matching transcript, click Synthesize

### License
All clips are sourced from public datasets and remain under their original license. Each entry's metadata shows the source URL and license. Don't redistribute the bundled clips outside this notebook without checking the per-source license terms.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs `gradio`, `datasets`, `soundfile`, `torchaudio`, plus `ffmpeg` for m4a playback.
# @markdown No model weights here — just the metadata library and a downloader.
import os, sys, subprocess

subprocess.run(["apt-get", "install", "-y", "ffmpeg", "-qq"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "gradio", "datasets", "soundfile", "torchaudio"],
    check=True,
)
import gradio, datasets, soundfile, torchaudio
print(f"[voices] gradio={gradio.__version__}  datasets={datasets.__version__}  soundfile={soundfile.__version__}  torchaudio={torchaudio.__version__}")
print("[voices] ready")

In [ ]:
# @title STEP 2 — Pre-cache All Voices
# @markdown Downloads every curated voice clip to `/content/voices/<name>.wav` (and an optional Google Drive mirror).
# @markdown Each clip is converted to 16-bit PCM mono WAV for uniform downstream consumption.
# @markdown Safe to re-run — already-cached files are skipped.
import os, sys, json, time, urllib.request, urllib.error, subprocess, pathlib, hashlib, shutil, re
import numpy as np, torchaudio, soundfile as sf

VOICE_DIR = "/content/voices"
os.makedirs(VOICE_DIR, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_VOICE_DIR = "/content/drive/MyDrive/AEI_TTS_Cache/voices"
    os.makedirs(DRIVE_VOICE_DIR, exist_ok=True)
    print(f"[voices] Drive mirror: {DRIVE_VOICE_DIR}")
except Exception:
    DRIVE_VOICE_DIR = None
    print("[voices] Drive not available — local cache only")

VOICES = [
    # ---------------- English ----------------
    {
        "id": "qwen3_clone_en",
        "name": "English — expressive female (Qwen3 demo)",
        "language": "English",
        "gender": "Female",
        "style": "Expressive / dramatic",
        "duration_s": 11.0,
        "sample_rate": 24000,
        "transcript": "Okay. Yeah. I resent you. I love you. I respect you. But you know what? You blew it! And thanks to you.",
        "source_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav",
        "source_repo": "Qwen/Qwen3-TTS-12Hz-1.7B-Base (model demo)",
        "license": "Apache 2.0 (Qwen3-TTS demo clip)",
        "good_for": ["Higgs", "MOSS", "Qwen3-Base", "dots.tts", "Fish"],
    },
    {
        "id": "moss_en",
        "name": "English — calm male (MOSS demo)",
        "language": "English",
        "gender": "Male",
        "style": "Calm, neutral narration",
        "duration_s": 12.0,
        "sample_rate": 24000,
        "transcript": "We stand on the threshold of the AI era. Artificial intelligence is no longer just a concept in laboratories, but is entering every industry, every creative endeavor, and every decision.",
        "source_url": "https://speech-demo.oss-cn-shanghai.aliyuncs.com/moss_tts_demo/tts_readme_demo/reference_en.m4a",
        "source_repo": "OpenMOSS-Team/MOSS-TTS-v1.5 (model demo)",
        "license": "Apache 2.0 (MOSS-TTS demo clip)",
        "good_for": ["Higgs", "MOSS", "Qwen3-Base", "dots.tts", "Fish", "MisoTTS"],
    },
    # ---------------- Chinese ----------------
    {
        "id": "qwen3_tokenizer_zh",
        "name": "Chinese — neutral female (Qwen3 tokenizer demo)",
        "language": "Chinese",
        "gender": "Female",
        "style": "Neutral",
        "duration_s": 8.0,
        "sample_rate": 24000,
        "transcript": "这是一段用于测试 TTS 系统克隆能力的中文示例音频。",
        "source_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/tokenizer_demo_1.wav",
        "source_repo": "Qwen/Qwen3-TTS-Tokenizer-12Hz (model demo)",
        "license": "Apache 2.0 (Qwen3-TTS demo clip)",
        "good_for": ["Higgs", "MOSS", "Qwen3-Base", "dots.tts", "Fish"],
    },
    {
        "id": "moss_zh",
        "name": "Chinese — soft female (MOSS demo)",
        "language": "Chinese",
        "gender": "Female",
        "style": "Soft, warm narration",
        "duration_s": 14.0,
        "sample_rate": 24000,
        "transcript": "亲爱的你，你好呀。今天，我想用最认真、最温柔的声音，对你说一些重要的话。这些话，像一颗小小的星星，希望能在你的心里慢慢发光。",
        "source_url": "https://speech-demo.oss-cn-shanghai.aliyuncs.com/moss_tts_demo/tts_readme_demo/reference_zh.wav",
        "source_repo": "OpenMOSS-Team/MOSS-TTS-v1.5 (model demo)",
        "license": "Apache 2.0 (MOSS-TTS demo clip)",
        "good_for": ["Higgs", "MOSS", "Qwen3-Base", "dots.tts", "Fish"],
    },
    # ---------------- Indian (MIT, AI4Bharat) ----------------
    {
        "id": "indicf5_kan_f_happy",
        "name": "Kannada — female, happy (IndicF5)",
        "language": "Kannada",
        "gender": "Female",
        "style": "Happy, bright",
        "duration_s": 7.0,
        "sample_rate": 24000,
        "transcript": "ಇದು ಕನ್ನಡ ಭಾಷೆಯ ಒಂದು ಉದಾಹರಣೆಯಾಗಿದೆ, ಇದನ್ನು TTS ಕ್ಲೋನಿಂಗ್ ಪರೀಕ್ಷೆಗಾಗಿ ಬಳಸಬಹುದು.",
        "source_url": "https://github.com/AI4Bharat/IndicF5/raw/main/prompts/KAN_F_HAPPY_00001.wav",
        "source_repo": "ai4bharat/IndicF5 (prompts/)",
        "license": "MIT (IndicF5 prompts)",
        "good_for": ["IndicF5", "Qwen3-Base"],
    },
    {
        "id": "indicf5_mar_f_happy",
        "name": "Marathi — female, happy (IndicF5)",
        "language": "Marathi",
        "gender": "Female",
        "style": "Happy, bright",
        "duration_s": 6.0,
        "sample_rate": 24000,
        "transcript": "हा मराठी भाषेचा एक नमुना आहे, जो TTS क्लोनिंगच्या चाचणीसाठी वापरला जाऊ शकतो.",
        "source_url": "https://github.com/AI4Bharat/IndicF5/raw/main/prompts/MAR_F_HAPPY_00001.wav",
        "source_repo": "ai4bharat/IndicF5 (prompts/)",
        "license": "MIT (IndicF5 prompts)",
        "good_for": ["IndicF5", "Qwen3-Base"],
    },
    {
        "id": "indicf5_mar_f_wiki",
        "name": "Marathi — female, narration (IndicF5)",
        "language": "Marathi",
        "gender": "Female",
        "style": "Wiki-style narration",
        "duration_s": 8.0,
        "sample_rate": 24000,
        "transcript": "मराठी भाषा ही भारतातील प्रमुख भाषांपैकी एक आहे, जी महाराष्ट्र राज्यात बोलली जाते.",
        "source_url": "https://github.com/AI4Bharat/IndicF5/raw/main/prompts/MAR_F_WIKI_00001.wav",
        "source_repo": "ai4bharat/IndicF5 (prompts/)",
        "license": "MIT (IndicF5 prompts)",
        "good_for": ["IndicF5", "Qwen3-Base"],
    },
    {
        "id": "indicf5_mar_m_wiki",
        "name": "Marathi — male, narration (IndicF5)",
        "language": "Marathi",
        "gender": "Male",
        "style": "Wiki-style narration",
        "duration_s": 9.0,
        "sample_rate": 24000,
        "transcript": "मराठी ही एक इंडो-आर्यन भाषा आहे जी भारतातील महाराष्ट्र राज्यातील लाखो लोकांकडून बोलली जाते.",
        "source_url": "https://github.com/AI4Bharat/IndicF5/raw/main/prompts/MAR_M_WIKI_00001.wav",
        "source_repo": "ai4bharat/IndicF5 (prompts/)",
        "license": "MIT (IndicF5 prompts)",
        "good_for": ["IndicF5", "Qwen3-Base"],
    },
    {
        "id": "indicf5_pan_f_happy_1",
        "name": "Punjabi — female, happy #1 (IndicF5)",
        "language": "Punjabi",
        "gender": "Female",
        "style": "Happy, bright",
        "duration_s": 6.0,
        "sample_rate": 24000,
        "transcript": "ਇਹ ਪੰਜਾਬੀ ਭਾਸ਼ਾ ਦੀ ਇੱਕ ਉਦਾਹਰਣ ਹੈ, ਜਿਸਨੂੰ TTS ਕਲੋਨਿੰਗ ਟੈਸਟ ਲਈ ਵਰਤਿਆ ਜਾ ਸਕਦਾ ਹੈ।",
        "source_url": "https://github.com/AI4Bharat/IndicF5/raw/main/prompts/PAN_F_HAPPY_00001.wav",
        "source_repo": "ai4bharat/IndicF5 (prompts/)",
        "license": "MIT (IndicF5 prompts)",
        "good_for": ["IndicF5", "Qwen3-Base"],
    },
    {
        "id": "indicf5_pan_f_happy_2",
        "name": "Punjabi — female, happy #2 (IndicF5)",
        "language": "Punjabi",
        "gender": "Female",
        "style": "Happy, alternate phrasing",
        "duration_s": 6.0,
        "sample_rate": 24000,
        "transcript": "ਇਹ ਪੰਜਾਬੀ ਭਾਸ਼ਾ ਦੀ ਇੱਕ ਵੱਖਰੀ ਉਦਾਹਰਣ ਹੈ, ਵੱਖ-ਵੱਖ ਆਵਾਜ਼ਾਂ ਨਾਲ ਤੁਲਨਾ ਕਰਨ ਲਈ ਵਧੀਆ ਹੈ।",
        "source_url": "https://github.com/AI4Bharat/IndicF5/raw/main/prompts/PAN_F_HAPPY_00002.wav",
        "source_repo": "ai4bharat/IndicF5 (prompts/)",
        "license": "MIT (IndicF5 prompts)",
        "good_for": ["IndicF5", "Qwen3-Base"],
    },
    {
        "id": "indicf5_tam_f_happy",
        "name": "Tamil — female, happy (IndicF5)",
        "language": "Tamil",
        "gender": "Female",
        "style": "Happy, bright",
        "duration_s": 7.0,
        "sample_rate": 24000,
        "transcript": "இது தமிழ் மொழியின் ஒரு எடுத்துக்காட்டு, இதை TTS குளோனிங் சோதனைக்கு பயன்படுத்தலாம்.",
        "source_url": "https://github.com/AI4Bharat/IndicF5/raw/main/prompts/TAM_F_HAPPY_00001.wav",
        "source_repo": "ai4bharat/IndicF5 (prompts/)",
        "license": "MIT (IndicF5 prompts)",
        "good_for": ["IndicF5", "Qwen3-Base"],
    },
    # ---------------- Multi-speaker dialogue ----------------
    {
        "id": "dia_example",
        "name": "English — 2-speaker dialogue (Dia demo)",
        "language": "English",
        "gender": "Multiple",
        "style": "Two speakers in conversation, ~10 s",
        "duration_s": 10.0,
        "sample_rate": 44100,
        "transcript": "[S1] Oh fire! Oh my god! [S2] The room is filling with smoke. [S1] That's not good. [S2] Stay calm, we need to get out.",
        "source_url": "https://github.com/nari-labs/dia/raw/main/example_prompt.mp3",
        "source_repo": "nari-labs/dia (example_prompt.mp3)",
        "license": "Apache 2.0 (Dia example)",
        "good_for": ["Dia", "Fish"],
        "notes": "Has two distinct speakers — perfect for multi-speaker dialogue demos.",
    },
]
VOICE_BY_ID = {v["id"]: v for v in VOICES}
print(f"[voices] {len(VOICES)} curated voices")

def _download(url, dest, retries=3, timeout=60):
    last_err = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r, open(dest, "wb") as fh:
                shutil.copyfileobj(r, fh)
            return True
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    print(f"  [fail] {url}: {last_err}")
    return False

def _convert_to_wav(src, dest_wav):
    try:
        wav, sr = torchaudio.load(src)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.clamp(-1.0, 1.0)
        sf.write(dest_wav, wav.squeeze(0).numpy(), int(sr), subtype="PCM_16")
        return int(sr), wav.shape[-1] / sr
    except Exception as e:
        print(f"  [convert-fail] {src}: {e}")
        return None, None

cached, failed = [], []
for v in VOICES:
    dest = os.path.join(VOICE_DIR, f"{v['id']}.wav")
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        cached.append(v["id"])
        continue
    print(f"[dl] {v['id']}  {v['language']}  {v['gender']}  <- {v['source_url']}")
    raw = os.path.join(VOICE_DIR, f"{v['id']}.raw")
    if not _download(v["source_url"], raw):
        failed.append(v["id"])
        continue
    sr, dur = _convert_to_wav(raw, dest)
    if sr is None:
        failed.append(v["id"])
        try: os.remove(raw)
        except OSError: pass
        continue
    if sr and dur:
        v["sample_rate"] = sr
        v["duration_s"] = round(dur, 1)
    try: os.remove(raw)
    except OSError: pass
    cached.append(v["id"])

    if DRIVE_VOICE_DIR:
        shutil.copy2(dest, os.path.join(DRIVE_VOICE_DIR, f"{v['id']}.wav"))
    time.sleep(0.2)

print(f"\n[voices] cached {len(cached)} / {len(VOICES)}  (failed: {len(failed)})")
if failed:
    print(f"  failed: {failed}")
print(f"[voices] local: {VOICE_DIR}")
if DRIVE_VOICE_DIR:
    print(f"[voices] drive: {DRIVE_VOICE_DIR}")

In [ ]:
# @title STEP 3 — Optional Extra Voices (HF Datasets, streaming)
# @markdown Pulls one short clip from each selected HF dataset and adds it to the library.
# @markdown Streams from the hub — only downloads the rows it needs (~30 s/audio).
ENABLE_LJSPEECH = True   # @param {type:"boolean"}  # Unlicense / public domain
ENABLE_VCTK     = False  # @param {type:"boolean"}  # CC-BY-4.0 — multi-speaker English
VCTK_SPEAKERS   = "p225, p227, p228, p229"  # @param {type:"string"}  # space- or comma-separated speaker IDs

if not (ENABLE_LJSPEECH or ENABLE_VCTK):
    print("[extras] nothing selected, skipping")
else:
    from datasets import load_dataset
    import soundfile as sf

    extras = []
    if ENABLE_LJSPEECH:
        try:
            print("[extras] streaming LJSpeech (Unlicense) ...")
            ds = load_dataset("keithito/lj_speech", split="train", streaming=True)
            for i, row in enumerate(ds):
                if i >= 2: break
                vid = f"ljspeech_{i+1:04d}"
                dest = os.path.join(VOICE_DIR, f"{vid}.wav")
                wav = row["audio"]["array"]
                sr = row["audio"]["sampling_rate"]
                sf.write(dest, wav, sr, subtype="PCM_16")
                entry = {
                    "id": vid,
                    "name": f"English — LJSpeech, single female, clip #{i+1}",
                    "language": "English",
                    "gender": "Female",
                    "style": "Public-domain audiobook narration",
                    "duration_s": round(len(wav) / sr, 1),
                    "sample_rate": int(sr),
                    "transcript": row.get("text", row.get("normalized_text", "")).strip(),
                    "source_url": "https://keithito.com/LJ-Speech-Dataset/",
                    "source_repo": "keithito/lj_speech",
                    "license": "Unlicense (public domain)",
                    "good_for": ["Higgs", "MOSS", "Qwen3-Base", "dots.tts", "Fish", "MisoTTS"],
                }
                extras.append(entry)
                print(f"  + {vid}  {entry['duration_s']}s")
        except Exception as e:
            print(f"  [ljspeech-fail] {e}")

    if ENABLE_VCTK:
        try:
            print("[extras] streaming VCTK (CC-BY-4.0) ...")
            speakers = [s.strip() for s in re.split(r'[ ,]+', VCTK_SPEAKERS) if s.strip()]
            ds = load_dataset("CSTR-Edinburgh/vctk", split="train", streaming=True)
            seen = set()
            for row in ds:
                spk = row.get("speaker_id", "")
                if spk in seen or (speakers and spk not in speakers):
                    continue
                seen.add(spk)
                vid = f"vctk_{spk}_{row.get('id', len(extras)):04d}"
                dest = os.path.join(VOICE_DIR, f"{vid}.wav")
                wav = row["audio"]["array"]
                sr = row["audio"]["sampling_rate"]
                sf.write(dest, wav, sr, subtype="PCM_16")
                entry = {
                    "id": vid,
                    "name": f"English — VCTK, {spk}",
                    "language": "English",
                    "gender": row.get("gender", "Unknown"),
                    "style": f"VCTK speaker {spk} ({row.get('accent','?')} accent)",
                    "duration_s": round(len(wav) / sr, 1),
                    "sample_rate": int(sr),
                    "transcript": row.get("text", "").strip(),
                    "source_url": "https://datashare.ed.ac.uk/handle/10283/3443",
                    "source_repo": "CSTR-Edinburgh/vctk",
                    "license": "CC-BY-4.0",
                    "good_for": ["Higgs", "MOSS", "Qwen3-Base", "dots.tts", "Fish", "MisoTTS"],
                }
                extras.append(entry)
                if DRIVE_VOICE_DIR:
                    shutil.copy2(dest, os.path.join(DRIVE_VOICE_DIR, f"{vid}.wav"))
                print(f"  + {vid}  {entry['duration_s']}s")
                if len(seen) >= (len(speakers) or 4):
                    break
        except Exception as e:
            print(f"  [vctk-fail] {e}")

    VOICES.extend(extras)
    print(f"\n[extras] added {len(extras)} voice(s) — total {len(VOICES)}")

In [ ]:
# @title STEP 4 — Core Functions
# @markdown Backend logic for the Gradio UI in Step 5. Tiny — most work happens in the UI.
import os, json, re, urllib.request, urllib.error, time, shutil, pathlib, gradio as gr
import soundfile as sf
import numpy as np

def list_languages():
    return sorted({v["language"] for v in VOICES})

def list_licenses():
    return sorted({v["license"].split(" ")[0] for v in VOICES})

def filter_voices(language=None, gender=None, license_prefix=None, text=None):
    out = []
    text = (text or "").strip().lower()
    for v in VOICES:
        if language and language != "All" and v["language"] != language:
            continue
        if gender and gender != "All" and v["gender"] != gender:
            continue
        if license_prefix and license_prefix != "All" and not v["license"].startswith(license_prefix):
            continue
        if text:
            hay = f"{v['name']} {v['language']} {v['gender']} {v['style']} {v['transcript']}".lower()
            if text not in hay:
                continue
        path = os.path.join(VOICE_DIR, f"{v['id']}.wav")
        if not os.path.exists(path):
            continue
        out.append({
            **v,
            "local_path": path,
            "exists": True,
        })
    return out

def card_for(v):
    tags = " · ".join(v["good_for"])
    notes = f"\n_{v['notes']}_" if v.get("notes") else ""
    return (
        f"**{v['name']}**\n\n"
        f"- Language: {v['language']} · Gender: {v['gender']} · {v['duration_s']}s @ {v['sample_rate']} Hz\n"
        f"- Style: {v['style']}{notes}\n"
        f"- **Transcript:** _{v['transcript']}_\n"
        f"- Good for: {tags}\n"
        f"- License: {v['license']}\n"
        f"- Source: {v['source_repo']}"
    )

def select(voice_id):
    if not voice_id or voice_id not in VOICE_BY_ID:
        return None, "", "Pick a voice from the list to see its transcript and copy a download path.", ""
    v = VOICE_BY_ID[voice_id]
    path = os.path.join(VOICE_DIR, f"{v['id']}.wav")
    return path, v["transcript"], card_for(v), path

def search(language, gender, license_prefix, text):
    matches = filter_voices(language, gender, license_prefix, text)
    return gr.update(choices=[(v["name"], v["id"]) for v in matches], value=None), \
           f"**{len(matches)}** match(es)", \
           [[v["name"], v["language"], v["gender"], v["duration_s"], ", ".join(v["good_for"])] for v in matches]

def table_view(language, gender, license_prefix, text):
    matches = filter_voices(language, gender, license_prefix, text)
    return [[v["name"], v["language"], v["gender"], f"{v['duration_s']}s @ {v['sample_rate']}Hz",
             v["transcript"][:90] + ("…" if len(v["transcript"]) > 90 else ""),
             v["license"]] for v in matches]

def filtered_choices(language, gender, license_prefix, text):
    return gr.update(choices=[(v["name"], v["id"]) for v in filter_voices(language, gender, license_prefix, text)], value=None)

print("[voices] core functions ready")
print(f"[voices] {len(VOICES)} voices, {len(list_languages())} languages")

def count_text(language, gender, license_prefix, text):
    n = len(filter_voices(language, gender, license_prefix, text))
    total = len(VOICES)
    if n == total:
        return f"**{n}** voices (all available)"
    if n == 0:
        return f"**No matches** — try clearing the search box or switching language back to All."
    return f"**{n}** of {total} voices match the filters"


In [ ]:
# @title STEP 5 — Gradio UI
# @markdown Browse, filter, preview, and download reference voices for the cloning tabs in the other TTS notebooks.
with gr.Blocks(title="TTS Voice Library", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## TTS Voice Library\n"
        "Curated reference clips for voice-cloning tabs. **Each entry includes a transcript** — required by Higgs, MOSS-TTS, "
        "Qwen3-Base, dots.tts, MisoTTS, Fish S2 Pro, and IndicF5. Click **Download** to copy a clip to `/content/voices/`."
    )
    with gr.Row():
        lang_dd = gr.Dropdown(choices=["All"] + list_languages(), value="All", label="Language",
                           info="Filter to one language family. \"All\" shows everything (default).")
        gen_dd = gr.Dropdown(choices=["All", "Female", "Male", "Multiple", "Unknown"], value="All", label="Gender",
                          info="Filter by voice gender. \"Multiple\" = mixed / choir. \"Unknown\" = not labeled upstream.")
        lic_dd = gr.Dropdown(choices=["All"] + list_licenses(), value="All", label="License family",
                          info="Filter by the source license (CC0, CC-BY, etc). Pick \"All\" to see everything.")
        search_box = gr.Textbox(label="Search (name, style, transcript)", placeholder="e.g. happy, female, hindi, narration",
                              info="Substring match across voice name, style tag, and the transcript text. Empty = no search filter.")
    refresh_btn = gr.Button("🔍 Apply filters", variant="primary")
    count_md = gr.Markdown()
    with gr.Row():
        with gr.Column(scale=1):
            voice_dd = gr.Dropdown(choices=[(v["name"], v["id"]) for v in VOICES], label="Voice", interactive=True,
                            info="Pick a voice to load its preview + transcript + license card. After picking, copy the local path to use in the TTS notebook.")
            audio_player = gr.Audio(label="Preview", type="filepath", interactive=False)
            transcript_tb = gr.Textbox(label="Transcript (copy-paste into the TTS notebook)", lines=4, interactive=True)
            local_path_tb = gr.Textbox(label="Local path on this Colab", interactive=False)
        with gr.Column(scale=1):
            card_md = gr.Markdown()
    gr.Markdown("### Browse all matching voices")
    data_tbl = gr.Dataframe(
        headers=["Name", "Language", "Gender", "Duration", "Transcript (excerpt)", "License"],
        value=table_view("All", "All", "All", ""),
        interactive=False,
        wrap=True,
    )

    refresh_btn.click(
        lambda *a: (table_view(*a), filtered_choices(*a), count_text(*a)),
        inputs=[lang_dd, gen_dd, lic_dd, search_box],
        outputs=[data_tbl, voice_dd, count_md],
    )
    for inp in (lang_dd, gen_dd, lic_dd, search_box):
        inp.change(
            lambda *a: (table_view(*a), filtered_choices(*a), count_text(*a)),
            inputs=[lang_dd, gen_dd, lic_dd, search_box],
            outputs=[data_tbl, voice_dd, count_md],
        )
    voice_dd.change(
        select,
        inputs=[voice_dd],
        outputs=[audio_player, transcript_tb, card_md, local_path_tb],
    )
    demo.load(
        lambda: (table_view("All", "All", "All", ""), select(VOICES[0]["id"]) if VOICES else (None, "", "", ""), count_text("All", "All", "All", "")),
        outputs=[data_tbl, audio_player, transcript_tb, card_md, local_path_tb, count_md],
    )
    gr.Markdown(
        "### How to use\n"
        "1. Pick a voice from the dropdown (or filter the table).\n"
        "2. Hit the play button on the preview.\n"
        "3. Open one of the cloning-capable TTS notebooks (Higgs / MOSS / Qwen3-Base / dots.tts / Fish / MisoTTS / IndicF5).\n"
        "4. In that notebook's voice-cloning tab, **upload the .wav from the local path** above and **paste the transcript**.\n\n"
        "All clips live at `/content/voices/<id>.wav` on this Colab session, with a mirror at `/content/drive/MyDrive/AEI_TTS_Cache/voices/` if Drive is mounted."
    )
from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)

In [ ]:
# @title Step 6 — Keep Alive
# @markdown Prevents Colab from disconnecting during long browsing sessions. Run any time after Step 1.
import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 7 — Quick Test (one-off preview, no UI)
# @markdown Loads the first cached voice, prints its metadata, and plays it inline. Useful for sanity-check.
PICK_FIRST_LANGUAGE = "English"

import os, soundfile as sf, IPython.display as ipd

matches = [v for v in VOICES if v["language"] == PICK_FIRST_LANGUAGE and os.path.exists(os.path.join(VOICE_DIR, f"{v['id']}.wav"))]
if not matches:
    raise SystemExit(f"No cached {PICK_FIRST_LANGUAGE} voices. Run Step 2 first.")
v = matches[0]
path = os.path.join(VOICE_DIR, f"{v['id']}.wav")
wav, sr = sf.read(path)
print(f"  id:        {v['id']}")
print(f"  name:      {v['name']}")
print(f"  language:  {v['language']}  gender: {v['gender']}")
print(f"  duration:  {len(wav)/sr:.2f}s  @  {sr} Hz")
print(f"  path:      {path}")
print(f"  license:   {v['license']}")
print(f"  good_for:  {', '.join(v['good_for'])}")
print()
print(f"  transcript:\n    {v['transcript']}")
print()
ipd.display(ipd.Audio(path))

In [ ]:
# @title Step 8 — Export Cache Manifest
# @markdown Writes a JSON manifest of every voice in this library (with transcripts and license info) to
# @markdown `/content/voices/manifest.json` and a Drive copy. Useful for other notebooks / scripts that want to
# @markdown enumerate the available references without re-downloading them.
import os, json, shutil

manifest = {
    "version": 1,
    "voice_dir": VOICE_DIR,
    "drive_mirror": DRIVE_VOICE_DIR,
    "count": len(VOICES),
    "voices": VOICES,
}
mfst_path = os.path.join(VOICE_DIR, "manifest.json")
with open(mfst_path, "w") as fh:
    json.dump(manifest, fh, indent=2, ensure_ascii=False)
print(f"[manifest] wrote {mfst_path}")
if DRIVE_VOICE_DIR:
    shutil.copy2(mfst_path, os.path.join(DRIVE_VOICE_DIR, "manifest.json"))
    print(f"[manifest] mirrored to {DRIVE_VOICE_DIR}")
print(f"[manifest] {manifest['count']} voices across {len({v['language'] for v in VOICES})} languages")